# 02 Behavioral EDA

**Purpose**

Understand how users, sessions, products, categories, brands, and prices differ across marketplace behaviors. This notebook should generate hypotheses for funnel and retention analysis, without becoming the final funnel or retention readout.


## Setup And Sample Load

This uses the same convenience sample pattern as `01_data_exploration.ipynb` for iteration. Final behavioral summaries should be rerun on the full dataset or a representative sample before making product recommendations.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.2f}".format)

DATA_DIR = Path("../data/raw")
RAW_FILES = sorted(DATA_DIR.glob("2019-*.csv"))
NROWS_PER_FILE = 100_000

RAW_FILES


In [ ]:
frames = []

for file_path in RAW_FILES:
    month_df = pd.read_csv(file_path, nrows=NROWS_PER_FILE)
    month_df["source_file"] = file_path.name
    frames.append(month_df)

df_raw = pd.concat(frames, ignore_index=True)

df_raw.head()


In [ ]:
df = df_raw.copy()

# Convert fields to analytical types.
df["event_time"] = pd.to_datetime(df["event_time"], utc=True)

id_columns = ["product_id", "category_id", "user_id", "user_session"]
df[id_columns] = df[id_columns].astype("string")

df["event_type"] = df["event_type"].astype("category")
df["category_code"] = df["category_code"].astype("string")
df["brand"] = df["brand"].astype("string")

df.dtypes


## Suggested Analysis Sections

- User behavior: events per user, sessions per user, products viewed per user
- Session behavior: session length, product/category breadth, cart and purchase presence
- Product behavior: most viewed/carted/purchased products
- Category and brand behavior: demand concentration and taxonomy coverage
- Price behavior: price distributions by event type and buyer/non-buyer behavior

Detailed funnel conversion belongs in `03_funnel_analysis.ipynb`. Retention definitions and cohort preparation belong in `04_retention_analysis.ipynb`.


In [ ]:
behavior_event_mix = (
    df.groupby("event_type", observed=False)
      .agg(
          events=("event_time", "size"),
          users=("user_id", "nunique"),
          sessions=("user_session", "nunique"),
          products=("product_id", "nunique"),
      )
      .reset_index()
      .sort_values("events", ascending=False)
)

behavior_event_mix


In [ ]:
session_behavior = (
    df.groupby("user_session")
      .agg(
          user_id=("user_id", "first"),
          first_event_time=("event_time", "min"),
          last_event_time=("event_time", "max"),
          events=("event_time", "size"),
          products=("product_id", "nunique"),
          categories=("category_id", "nunique"),
          brands=("brand", "nunique"),
      )
      .reset_index()
)

session_behavior["session_duration_minutes"] = (
    session_behavior["last_event_time"] - session_behavior["first_event_time"]
).dt.total_seconds() / 60

session_behavior.describe().round(2)


## Behavioral EDA Handoff

Use this notebook to identify segments and hypotheses worth testing in funnel and retention analysis. Keep final conversion and retention claims in notebooks `03` and `04`.
